# Desenvolvimento de Feature Store - Products


Ideias

- Ranking de categorias do seller_id dentro das janelas

In [0]:
DECLARE VARIABLE deploy_date DATE = '2018-07-01'

In [0]:
-- 1 Tabela base de pedidos com filtro de safra embutido
WITH tb_pedidos AS (
  SELECT *
  FROM workspace.olist.orders
  WHERE order_purchase_timestamp < deploy_date
),

-- 2 Tabela base de itens
tb_itens AS (
  SELECT
    i.seller_id
    , o.order_id
    , o.order_purchase_timestamp
    , i.order_item_id
    , IFNULL(p.product_category_name, 'NA') AS product_category_name
    , i.product_id
    , i.price
    , i.freight_value
  FROM tb_pedidos AS o
  INNER JOIN olist.order_items AS i
    ON i.order_id = o.order_id
  LEFT JOIN olist.products AS p
    ON p.product_id = i.product_id
),

-- 3 Listagem de seller_id e product_category_name. Flag 1 caso tenha tido vendas na janela, caso contrário 0.
tb_seller_category_list (
    SELECT
        seller_id
        , product_category_name
        , MAX(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 14) THEN 1 ELSE 0 END) AS hadCatSale14d
        , MAX(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 28) THEN 1 ELSE 0 END) AS hadCatSale28d
        , MAX(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 56) THEN 1 ELSE 0 END) AS hadCatSale56d
        , MAX(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 365) THEN 1 ELSE 0 END) AS hadCatSale365d
        , 1 AS hadCatSaleVida
    FROM tb_itens
    GROUP BY seller_id, product_category_name
    ORDER BY seller_id, product_category_name
),

-- 4 Listagem de seller_id e product_id. Flag 1 caso tenha tido vendas na janela, caso contrário 0.
tb_seller_product_list (
    SELECT
        seller_id
        , product_id
        , MAX(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 14) THEN 1 ELSE 0 END) AS hadProdSale14d
        , MAX(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 28) THEN 1 ELSE 0 END) AS hadProdSale28d
        , MAX(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 56) THEN 1 ELSE 0 END) AS hadProdSale56d
        , MAX(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 365) THEN 1 ELSE 0 END) AS hadProdSale365d
        , 1 AS hadProdSaleVida
    FROM tb_itens
    GROUP BY seller_id, product_id
    ORDER BY seller_id, product_id
),

-- 5 Contagem de vendedores distintos por categoria
tb_category_window AS (
  SELECT
    product_category_name
    , IFNULL(COUNT(DISTINCT CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 14) THEN SELLER_ID END), 0) AS ctDistinctCatSellers14d
    , IFNULL(COUNT(DISTINCT CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 28) THEN SELLER_ID END), 0) AS ctDistinctCatSellers28d
    , IFNULL(COUNT(DISTINCT CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 56) THEN SELLER_ID END), 0) AS ctDistinctCatSellers56d
    , IFNULL(COUNT(DISTINCT CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 365) THEN SELLER_ID END), 0) AS ctDistinctCatSellers365d
    , IFNULL(COUNT(DISTINCT SELLER_ID), 0) AS ctDistinctCatSellersVida
  FROM tb_itens
  GROUP BY product_category_name
),

-- 6 Contagem de vendedores distintos por produto
tb_product_window AS (
  SELECT
    product_id
    , IFNULL(COUNT(DISTINCT CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 14) THEN SELLER_ID END), 0) AS ctDistinctProdSellers14d
    , IFNULL(COUNT(DISTINCT CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 28) THEN SELLER_ID END), 0) AS ctDistinctProdSellers28d
    , IFNULL(COUNT(DISTINCT CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 56) THEN SELLER_ID END), 0) AS ctDistinctProdSellers56d
    , IFNULL(COUNT(DISTINCT CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 365) THEN SELLER_ID END), 0) AS ctDistinctProdSellers365d
    , IFNULL(COUNT(DISTINCT SELLER_ID), 0) AS ctDistinctProdSellersVida
  FROM tb_itens
  GROUP BY product_id
),

-- 7 Contagem de concorrentes de categorias por seller_id (3 + 5)
tb_category_competitity (
    SELECT
        t1.seller_id
        , SUM(ctDistinctCatSellers14d - hadCatSale14d) AS vlContagemCategoriaConcorrentesD14
        , SUM(ctDistinctCatSellers28d - hadCatSale28d) AS vlContagemCategoriaConcorrentesD28
        , SUM(ctDistinctCatSellers56d - hadCatSale56d) AS vlContagemCategoriaConcorrentesD56
        , SUM(ctDistinctCatSellers365d - hadCatSale365d) AS vlContagemCategoriaConcorrentesD365
        , SUM(ctDistinctCatSellersVida - hadCatSaleVida) AS vlContagemCategoriaConcorrentesVida
    FROM tb_seller_category_list AS t1
    LEFT JOIN tb_category_window AS t2
        ON t1.product_category_name = t2.product_category_name
    GROUP BY t1.seller_id
),

-- 8 Contagem de concorrentes de produtos por seller_id (4 + 6)
tb_product_competitity (
    SELECT
        t1.seller_id
        , SUM(ctDistinctProdSellers14d - hadProdSale14d) AS vlContagemProdutosConcorrentesD14
        , SUM(ctDistinctProdSellers28d - hadProdSale28d) AS vlContagemProdutosConcorrentesD28
        , SUM(ctDistinctProdSellers56d - hadProdSale56d) AS vlContagemProdutosConcorrentesD56
        , SUM(ctDistinctProdSellers365d - hadProdSale365d) AS vlContagemProdutosConcorrentesD365
        , SUM(ctDistinctProdSellersVida - hadProdSaleVida) AS vlContagemProdutosConcorrentesVida
    FROM tb_seller_product_list AS t1
    LEFT JOIN tb_product_window AS t2
        ON t1.product_id = t2.product_id
    GROUP BY t1.seller_id
),

-- 9 Contagem de quantidade de categorias e produtos distintos nas janelas
tb_seller_cat_prod_count (
    SELECT
        seller_id
        , COUNT(DISTINCT CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 14) THEN product_category_name END) AS vlCategoriasDistintasD14
        , COUNT(DISTINCT CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 28) THEN product_category_name END) AS vlCategoriasDistintasD28
        , COUNT(DISTINCT CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 56) THEN product_category_name END) AS vlCategoriasDistintasD56
        , COUNT(DISTINCT CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 365) THEN product_category_name END) AS vlCategoriasDistintasD365
        , COUNT(DISTINCT product_category_name) AS vlCategoriasDistintasVida
        , COUNT(DISTINCT CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 14) THEN product_id END) AS vlProdutosDistintosD14
        , COUNT(DISTINCT CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 28) THEN product_id END) AS vlProdutosDistintosD28
        , COUNT(DISTINCT CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 56) THEN product_id END) AS vlProdutosDistintosD56
        , COUNT(DISTINCT CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 365) THEN product_id END) AS vlProdutosDistintosD365
        , COUNT(DISTINCT product_id) AS vlProdutosDistintosVida
    FROM tb_itens
    GROUP BY seller_id
),

-- 10 Listagem de produtos distintos vendidos pelos seller_id
tb_distinct_seller_prod_description AS (
    SELECT DISTINCT
        t1.seller_id
        , t1.product_id
        , IFNULL(t2.product_description_lenght, 0) AS product_description_length
        , IFNULL(t2.product_photos_qty, 0) AS product_photos_qty
    FROM tb_itens AS t1
    LEFT JOIN olist.products AS t2
        ON t1.product_id = t2.product_id
),

-- 11 Cálculo de métricas de descrição e fotos de produtos (10)
tb_atributos_produtos AS (
    SELECT
        seller_id
        , MEAN(product_description_length) AS vlMediaCaracteresDescricao
        , MIN(product_description_length) AS vlMinCaracteresDescricao
        , PERCENTILE(product_description_length, 0.25) AS vlP25CaracteresDescricao
        , MEDIAN(product_description_length) AS vlMedianaCaracteresDescricao
        , PERCENTILE(product_description_length, 0.75) AS vlP75CaracteresDescricao
        , MAX(product_description_length) AS vlMaxCaracteresDescricao
        , MEAN(product_photos_qty) AS vlMediaFotosProduto
    FROM tb_distinct_seller_prod_description
    GROUP BY seller_id
),

-- 12 Listagem de peso em kg e cubagem em cm3 dos pedidos de cada item
tb_seller_prod_dimensions_and_weight AS (
    SELECT
        t1.seller_id
        , order_purchase_timestamp
        , t1.product_id
        , t2.product_weight_g / 1000 AS product_weight_kg
        , (t2.product_length_cm / 1) * (t2.product_height_cm / 1) * (t2.product_width_cm / 1) AS product_cubic_volume_cm3
    FROM tb_itens AS t1
    LEFT JOIN olist.products AS t2
        ON t1.product_id = t2.product_id
),

-- 13 Cálculo de métricas de peso e cubagem de produtos (12)
tb_peso_e_cubagem_dos_produtos AS (
    SELECT
        seller_id
        , MEAN(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 14) THEN product_weight_kg END) AS vlMediaPesoProdutoD14
        , MEAN(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 28) THEN product_weight_kg END) AS vlMediaPesoProdutoD28
        , MEAN(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 56) THEN product_weight_kg END) AS vlMediaPesoProdutoD56
        , MEAN(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 365) THEN product_weight_kg END) AS vlMediaPesoProdutoD365
        , MEAN(product_weight_kg) AS vlMediaPesoProdutoVida
        , MIN(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 14) THEN product_weight_kg END) AS vlMinPesoProdutoD14
        , MIN(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 28) THEN product_weight_kg END) AS vlMinPesoProdutoD28
        , MIN(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 56) THEN product_weight_kg END) AS vlMinPesoProdutoD56
        , MIN(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 365) THEN product_weight_kg END) AS vlMinPesoProdutoD365
        , MIN(product_weight_kg) AS vlMinPesoProdutoVida
        , PERCENTILE(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 14) THEN product_weight_kg END, 0.25) AS vlP25PesoProdutoD14
        , PERCENTILE(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 28) THEN product_weight_kg END, 0.25) AS vlP25PesoProdutoD28
        , PERCENTILE(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 56) THEN product_weight_kg END, 0.25) AS vlP25PesoProdutoD56
        , PERCENTILE(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 365) THEN product_weight_kg END, 0.25) AS vlP25PesoProdutoD365
        , PERCENTILE(product_weight_kg, 0.25) AS vlP25PesoProdutoVida
        , MEDIAN(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 14) THEN product_weight_kg END) AS vlMediaPesoProdutoD14
        , MEDIAN(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 28) THEN product_weight_kg END) AS vlMediaPesoProdutoD28
        , MEDIAN(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 56) THEN product_weight_kg END) AS vlMediaPesoProdutoD56
        , MEDIAN(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 365) THEN product_weight_kg END) AS vlMediaPesoProdutoD365
        , MEDIAN(product_weight_kg) AS vlMediaPesoProdutoVida
        , PERCENTILE(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 14) THEN product_weight_kg END, 0.75) AS vlP75PesoProdutoD14
        , PERCENTILE(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 28) THEN product_weight_kg END, 0.75) AS vlP75PesoProdutoD28
        , PERCENTILE(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 56) THEN product_weight_kg END, 0.75) AS vlP75PesoProdutoD56
        , PERCENTILE(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 365) THEN product_weight_kg END, 0.75) AS vlP75PesoProdutoD365
        , PERCENTILE(product_weight_kg, 0.25) AS vlP75PesoProdutoVida
        , MAX(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 14) THEN product_weight_kg END) AS vlMaxPesoProdutoD14
        , MAX(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 28) THEN product_weight_kg END) AS vlMaxPesoProdutoD28
        , MAX(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 56) THEN product_weight_kg END) AS vlMaxPesoProdutoD56
        , MAX(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 365) THEN product_weight_kg END) AS vlMaxPesoProdutoD365
        , MAX(product_weight_kg) AS vlMaxPesoProdutoVida
        , SUM(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 14) THEN product_weight_kg END) AS vlTotalPesoProdutoD14
        , SUM(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 28) THEN product_weight_kg END) AS vlTotalPesoProdutoD28
        , SUM(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 56) THEN product_weight_kg END) AS vlTotalPesoProdutoD56
        , SUM(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 365) THEN product_weight_kg END) AS vlTotalPesoProdutoD365
        , SUM(product_weight_kg) AS vlTotalPesoProdutoVida

        , MEAN(CASE WHEN order_purchase_timestamp > DATE_SUB(deploy_date, 14) THEN product_weight_kg END) AS vlMediaCubagemProdutoD14
        , MEAN(CASE WHEN order_purchase_timestamp > DATE_SUB(deploy_date, 28) THEN product_weight_kg END) AS vlMediaCubagemProdutoD28
        , MEAN(CASE WHEN order_purchase_timestamp > DATE_SUB(deploy_date, 56) THEN product_weight_kg END) AS vlMediaCubagemProdutoD56
        , MEAN(CASE WHEN order_purchase_timestamp > DATE_SUB(deploy_date, 365) THEN product_weight_kg END) AS vlMediaCubagemProdutoD365
        , MEAN(product_weight_kg) AS vlMediaCubagemProdutoVida
        , SUM(CASE WHEN order_purchase_timestamp > DATE_SUB(deploy_date, 14) THEN product_weight_kg END) AS vlTotalCubagemProdutoD14
        , SUM(CASE WHEN order_purchase_timestamp > DATE_SUB(deploy_date, 28) THEN product_weight_kg END) AS vlTotalCubagemProdutoD28
        , SUM(CASE WHEN order_purchase_timestamp > DATE_SUB(deploy_date, 56) THEN product_weight_kg END) AS vlTotalCubagemProdutoD56
        , SUM(CASE WHEN order_purchase_timestamp > DATE_SUB(deploy_date, 365) THEN product_weight_kg END) AS vlTotalCubagemProdutoD365
        , SUM(product_weight_kg) AS vlTotalCubagemProdutoVida
    FROM tb_seller_prod_dimensions_and_weight
    GROUP BY seller_id
),

-- 14 Receita e frete por janela
tb_receita_e_frete_por_janela AS (
  SELECT
    seller_id
    , SUM(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 14) THEN price ELSE 0 END) AS vlReceitaTotD14
    , SUM(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 28) THEN price ELSE 0 END) AS vlReceitaTotD28
    , SUM(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 56) THEN price ELSE 0 END) AS vlReceitaTotD56
    , SUM(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 365) THEN price ELSE 0 END) AS vlReceitaTotD365
    , SUM(price) AS vlReceitaTotVida
    , SUM(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 14) THEN freight_value ELSE 0 END) AS vlFreteTotD14
    , SUM(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 28) THEN freight_value ELSE 0 END) AS vlFreteTotD28
    , SUM(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 56) THEN freight_value ELSE 0 END) AS vlFreteTotD56
    , SUM(CASE WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 365) THEN freight_value ELSE 0 END) AS vlFreteTotD365
    , SUM(freight_value) AS vlFreteTotVida
  FROM tb_itens
  GROUP BY seller_id
),
-- 15 tb_indicadores_por_kg (14)
tb_indicadores_por_kg AS (
  SELECT
    t1.seller_id
    , t1.vlReceitaTotD14 / t2.vlTotalPesoProdutoD14 AS vlPrecoKgD14
    , t1.vlReceitaTotD28 / t2.vlTotalPesoProdutoD28 AS vlPrecoKgD28
    , t1.vlReceitaTotD56 / t2.vlTotalPesoProdutoD56 AS vlPrecoKgD56
    , t1.vlReceitaTotD365 / t2.vlTotalPesoProdutoD365 AS vlPrecoKgD365
    , t1.vlReceitaTotVida / t2.vlTotalPesoProdutoVida AS vlPrecoKgVida
    , t1.vlFreteTotD14 / t2.vlTotalPesoProdutoD14 AS vlFreteKgD14
    , t1.vlFreteTotD28 / t2.vlTotalPesoProdutoD28 AS vlFreteKgD28
    , t1.vlFreteTotD56 / t2.vlTotalPesoProdutoD56 AS vlFreteKgD56
    , t1.vlFreteTotD365 / t2.vlTotalPesoProdutoD365 AS vlFreteKgD365
    , t1.vlFreteTotVida / t2.vlTotalPesoProdutoVida AS vlFreteKgVida
  FROM tb_receita_e_frete_por_janela AS t1
  LEFT JOIN tb_peso_e_cubagem_dos_produtos AS t2
    ON t1.seller_id = t2.seller_id
)

SELECT *
FROM tb_indicadores_por_kg
LIMIT 100;

In [0]:
WITH params AS (
  SELECT DATE '2018-07-01' AS deploy_date
),

tb_pedidos AS (
  SELECT
    o.*,
    p.deploy_date
  FROM workspace.olist.orders AS o
  CROSS JOIN params AS p
  WHERE o.order_purchase_timestamp < p.deploy_date
),

tb_itens AS (
  SELECT
    i.seller_id,
    o.order_id,
    o.order_purchase_timestamp,
    i.order_item_id,
    IFNULL(p.product_category_name, 'NA') AS product_category_name,
    i.product_id,
    i.price,
    i.freight_value,
    o.deploy_date
  FROM tb_pedidos AS o
  INNER JOIN olist.order_items AS i
    ON i.order_id = o.order_id
  LEFT JOIN olist.products AS p
    ON p.product_id = i.product_id
),

tb_sellers AS (
  SELECT DISTINCT
    seller_id
  FROM tb_itens
),

tb_cat_receita AS (
  SELECT
    seller_id,
    product_category_name,

    SUM(
      CASE 
        WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 14)
        THEN COALESCE(price, 0)
        ELSE 0
      END
    ) AS vlTotalReceitaD14,

    SUM(
      CASE 
        WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 28)
        THEN COALESCE(price, 0)
        ELSE 0
      END
    ) AS vlTotalReceitaD28,

    SUM(
      CASE 
        WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 56)
        THEN COALESCE(price, 0)
        ELSE 0
      END
    ) AS vlTotalReceitaD56,

    SUM(
      CASE 
        WHEN order_purchase_timestamp >= DATE_SUB(deploy_date, 365)
        THEN COALESCE(price, 0)
        ELSE 0
      END
    ) AS vlTotalReceitaD365,

    SUM(COALESCE(price, 0)) AS vlTotalReceitaVida

  FROM tb_itens
  GROUP BY seller_id, product_category_name
),

tb_cat_janela AS (
  SELECT seller_id, product_category_name, 'D14' AS janela, vlTotalReceitaD14 AS vlReceitaCategoria
  FROM tb_cat_receita

  UNION ALL

  SELECT seller_id, product_category_name, 'D28' AS janela, vlTotalReceitaD28 AS vlReceitaCategoria
  FROM tb_cat_receita

  UNION ALL

  SELECT seller_id, product_category_name, 'D56' AS janela, vlTotalReceitaD56 AS vlReceitaCategoria
  FROM tb_cat_receita

  UNION ALL

  SELECT seller_id, product_category_name, 'D365' AS janela, vlTotalReceitaD365 AS vlReceitaCategoria
  FROM tb_cat_receita

  UNION ALL

  SELECT seller_id, product_category_name, 'Vida' AS janela, vlTotalReceitaVida AS vlReceitaCategoria
  FROM tb_cat_receita
),

tb_rank_base AS (
  SELECT
    seller_id,
    product_category_name,
    janela,
    vlReceitaCategoria,

    SUM(vlReceitaCategoria) OVER (
      PARTITION BY seller_id, janela
    ) AS vlReceitaSellerJanela

  FROM tb_cat_janela
  WHERE vlReceitaCategoria > 0
),

tb_rank AS (
  SELECT
    seller_id,
    product_category_name,
    janela,
    vlReceitaCategoria,
    vlReceitaSellerJanela,

    CAST(vlReceitaCategoria AS DOUBLE) / vlReceitaSellerJanela AS shareReceitaCategoria,

    ROW_NUMBER() OVER (
      PARTITION BY seller_id, janela
      ORDER BY vlReceitaCategoria DESC, product_category_name ASC
    ) AS nrRankCategoria

  FROM tb_rank_base
)

SELECT
  s.seller_id,

  MAX(CASE WHEN r.janela = 'D14' AND r.nrRankCategoria = 1 THEN r.product_category_name END) AS descTopCategoria1D14,
  MAX(CASE WHEN r.janela = 'D14' AND r.nrRankCategoria = 2 THEN r.product_category_name END) AS descTopCategoria2D14,
  MAX(CASE WHEN r.janela = 'D14' AND r.nrRankCategoria = 3 THEN r.product_category_name END) AS descTopCategoria3D14,

  MAX(CASE WHEN r.janela = 'D28' AND r.nrRankCategoria = 1 THEN r.product_category_name END) AS descTopCategoria1D28,
  MAX(CASE WHEN r.janela = 'D28' AND r.nrRankCategoria = 2 THEN r.product_category_name END) AS descTopCategoria2D28,
  MAX(CASE WHEN r.janela = 'D28' AND r.nrRankCategoria = 3 THEN r.product_category_name END) AS descTopCategoria3D28,

  MAX(CASE WHEN r.janela = 'D56' AND r.nrRankCategoria = 1 THEN r.product_category_name END) AS descTopCategoria1D56,
  MAX(CASE WHEN r.janela = 'D56' AND r.nrRankCategoria = 2 THEN r.product_category_name END) AS descTopCategoria2D56,
  MAX(CASE WHEN r.janela = 'D56' AND r.nrRankCategoria = 3 THEN r.product_category_name END) AS descTopCategoria3D56,

  MAX(CASE WHEN r.janela = 'D365' AND r.nrRankCategoria = 1 THEN r.product_category_name END) AS descTopCategoria1D365,
  MAX(CASE WHEN r.janela = 'D365' AND r.nrRankCategoria = 2 THEN r.product_category_name END) AS descTopCategoria2D365,
  MAX(CASE WHEN r.janela = 'D365' AND r.nrRankCategoria = 3 THEN r.product_category_name END) AS descTopCategoria3D365,

  MAX(CASE WHEN r.janela = 'Vida' AND r.nrRankCategoria = 1 THEN r.product_category_name END) AS descTopCategoria1Vida,
  MAX(CASE WHEN r.janela = 'Vida' AND r.nrRankCategoria = 2 THEN r.product_category_name END) AS descTopCategoria2Vida,
  MAX(CASE WHEN r.janela = 'Vida' AND r.nrRankCategoria = 3 THEN r.product_category_name END) AS descTopCategoria3Vida,

  MAX(CASE WHEN r.janela = 'D14' AND r.nrRankCategoria = 1 THEN r.shareReceitaCategoria END) AS shareTopCategoria1D14,
  MAX(CASE WHEN r.janela = 'D14' AND r.nrRankCategoria = 2 THEN r.shareReceitaCategoria END) AS shareTopCategoria2D14,
  MAX(CASE WHEN r.janela = 'D14' AND r.nrRankCategoria = 3 THEN r.shareReceitaCategoria END) AS shareTopCategoria3D14,

  MAX(CASE WHEN r.janela = 'D28' AND r.nrRankCategoria = 1 THEN r.shareReceitaCategoria END) AS shareTopCategoria1D28,
  MAX(CASE WHEN r.janela = 'D28' AND r.nrRankCategoria = 2 THEN r.shareReceitaCategoria END) AS shareTopCategoria2D28,
  MAX(CASE WHEN r.janela = 'D28' AND r.nrRankCategoria = 3 THEN r.shareReceitaCategoria END) AS shareTopCategoria3D28,

  MAX(CASE WHEN r.janela = 'D56' AND r.nrRankCategoria = 1 THEN r.shareReceitaCategoria END) AS shareTopCategoria1D56,
  MAX(CASE WHEN r.janela = 'D56' AND r.nrRankCategoria = 2 THEN r.shareReceitaCategoria END) AS shareTopCategoria2D56,
  MAX(CASE WHEN r.janela = 'D56' AND r.nrRankCategoria = 3 THEN r.shareReceitaCategoria END) AS shareTopCategoria3D56,

  MAX(CASE WHEN r.janela = 'D365' AND r.nrRankCategoria = 1 THEN r.shareReceitaCategoria END) AS shareTopCategoria1D365,
  MAX(CASE WHEN r.janela = 'D365' AND r.nrRankCategoria = 2 THEN r.shareReceitaCategoria END) AS shareTopCategoria2D365,
  MAX(CASE WHEN r.janela = 'D365' AND r.nrRankCategoria = 3 THEN r.shareReceitaCategoria END) AS shareTopCategoria3D365,

  MAX(CASE WHEN r.janela = 'Vida' AND r.nrRankCategoria = 1 THEN r.shareReceitaCategoria END) AS shareTopCategoria1Vida,
  MAX(CASE WHEN r.janela = 'Vida' AND r.nrRankCategoria = 2 THEN r.shareReceitaCategoria END) AS shareTopCategoria2Vida,
  MAX(CASE WHEN r.janela = 'Vida' AND r.nrRankCategoria = 3 THEN r.shareReceitaCategoria END) AS shareTopCategoria3Vida

FROM tb_sellers AS s
LEFT JOIN tb_rank AS r
  ON s.seller_id = r.seller_id
  AND r.nrRankCategoria <= 3
GROUP BY s.seller_id;